# Подготовка данных для сайта «Steam Games Market»

Курсовая работа: «Разработка веб-сайта рынка видеоигр Steam».
Источник: Kaggle, «Steam Games Dataset 2025» (автор artermiloff),
https://www.kaggle.com/datasets/artermiloff/steam-games-dataset

Ноутбук делает всю предобработку и сохраняет три файла в `data/`:
`games.json`, `prices.json`, `platforms.json`.
Запускать сверху вниз: **Runtime → Restart and run all**.

**Структура**
1. Загрузка набора
2. Разведочный анализ (что лежит в данных, в каком формате)
3. Настройки: имена столбцов и правила отбора
4. Очистка и воронка отбора
5. Стратифицированная выборка
6. Признаки и агрегаты
7. Экспорт JSON и контроль

Выборка **не отражает весь каталог Steam**: она нужна, чтобы показать закономерности на данных.

## 1. Загрузка набора

In [ ]:
import kagglehub, os, glob

path = kagglehub.dataset_download("artermiloff/steam-games-dataset")
print("Папка набора:", path)
for f in sorted(glob.glob(os.path.join(path, "**", "*"), recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f) / 1024 / 1024:9.1f} МБ  {os.path.relpath(f, path)}")

Исходные файлы большие (сотни МБ), поэтому читаем их аккуратно:
сначала только заголовки, потом только нужные столбцы.

In [ ]:
import pandas as pd

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
SRC = max(csv_files, key=os.path.getsize)   # основной файл — самый большой
print("Основной файл:", SRC)

all_cols = pd.read_csv(SRC, nrows=0).columns.tolist()
print(len(all_cols), "столбцов:")
print(all_cols)

**Сводка по всем столбцам.** Прочитает первые 20 000 строк и покажет для каждого столбца тип,
долю заполненных значений и пример. **Скопируй вывод этой ячейки целиком и пришли его** — по нему
я сверю имена столбцов и настрою фильтры.

In [ ]:
head = pd.read_csv(SRC, nrows=20000, low_memory=False)
for c in head.columns:
    filled = head[c].notna().mean()
    vals = head[c].dropna()
    example = str(vals.iloc[0])[:50].replace("\n", " ") if len(vals) else ""
    print(f"{c} | {head[c].dtype} | заполнено {filled:.0%} | {example}")

## 2. Разведочный анализ

Сначала проверяем, есть ли столбцы, которые названы в требованиях (и какие ещё могут пригодиться).

In [ ]:
wanted = ["appid", "name", "release_date", "required_age", "price", "discount",
          "windows", "mac", "linux", "supported_languages", "full_audio_languages",
          "genres", "categories", "tags", "header_image", "screenshots",
          "positive", "negative", "pct_pos_total", "num_reviews_total",
          "average_playtime_forever", "median_playtime_forever", "estimated_owners",
          "is_free", "dlc_count", "short_description", "type"]
for c in wanted:
    print(("есть   " if c in all_cols else "НЕТ    ") + c)

print()
print("Похожие по смыслу на «скидка / время / скриншот / обложка»:")
print([c for c in all_cols if any(k in c.lower() for k in ("disc", "playtime", "screen", "header", "image", "cover"))])
print("Похожие на «возраст / взрослый / free»:")
print([c for c in all_cols if any(k in c.lower() for k in ("age", "adult", "sexual", "mature", "descriptor", "free"))])

In [ ]:
# Читаем только нужное (экономия памяти); недостающие столбцы просто пропускаем
use = [c for c in wanted if c in all_cols and c != "estimated_owners"]
raw = pd.read_csv(SRC, usecols=use, low_memory=False)
print(raw.shape)
raw.info(memory_usage="deep")

In [ ]:
# Пропуски и дубликаты
print("Дубликаты по appid:", raw.duplicated("appid").sum() if "appid" in raw else "нет appid")
print("Дубликаты по названию:", raw.duplicated("name").sum())
print()
print((raw.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())

In [ ]:
# Как выглядят значения: смотрим на реальные примеры, ничего не предполагая
pd.set_option("display.max_colwidth", 120)
print(raw.sample(5, random_state=1).T.to_string())

In [ ]:
# Дата выхода хранится текстом: какие бывают форматы?
s = raw["release_date"].dropna().astype(str)
print(s.sample(10, random_state=1).tolist())
print("Распознано как дата:", pd.to_datetime(s, errors="coerce").notna().mean().round(3))
print("Нераспознанные примеры:", s[pd.to_datetime(s, errors="coerce").isna()].head(10).tolist())

In [ ]:
# Цена: единицы измерения и бесплатные игры; и как связаны цена и скидка
print(raw["price"].describe().round(2).to_string())
print("Бесплатных (price == 0):", (raw["price"] == 0).sum())
if "discount" in raw:
    d = raw[raw["discount"].fillna(0) > 0]
    print("Игр со скидкой:", len(d))
    print(d[["name", "price", "discount"]].head(10).to_string())
    print("\nПРОВЕРЬ ВРУЧНУЮ на сайте Steam 2–3 игры из списка выше: показана ли в price цена со скидкой "
          "или без. Решение запиши в README.")

In [ ]:
# Контрольные игры: сравни цену с той, что видишь на странице Steam, чтобы понять,
# записана ли в price цена уже со скидкой (и что значит discount)
chk = raw[raw["name"].astype(str).str.contains("Rainbow Six|Garry's Mod|^Among Us$|Witcher 3|Grand Theft Auto V|Terraria", regex=True)]
print(chk[["name", "price"] + (["discount"] if "discount" in raw else [])].head(12).to_string())

In [ ]:
# Языки, жанры, теги: в каком формате лежат?
for c in ["supported_languages", "full_audio_languages", "genres", "categories", "tags", "screenshots"]:
    if c in raw:
        print(c, "->", repr(raw[c].dropna().iloc[0])[:200])

In [ ]:
# Отзывы: сверка pct_pos_total с расчётом по positive / (positive + negative)
if {"positive", "negative", "pct_pos_total"} <= set(raw.columns):
    tot = raw["positive"] + raw["negative"]
    calc = (raw["positive"] / tot * 100).where(tot > 0)
    diff = (calc - raw["pct_pos_total"]).abs()
    both = tot.gt(0) & raw["num_reviews_total"].gt(0)
    print("Игр с отзывами в обоих источниках:", int(both.sum()))
    print("Медианная |разница| доли, п.п.:", round(float(diff[both].median()), 2))
    print("Расхождение доли > 2 п.п.:", int((diff[both] > 2).sum()), "из", int(both.sum()))
    ratio = (raw["num_reviews_total"] / tot)[both]
    print("num_reviews_total / (positive + negative): медиана", round(float(ratio.median()), 2),
          "квартили", ratio.quantile([.25, .75]).round(2).tolist())
    print(raw.loc[both, ["name", "positive", "negative", "num_reviews_total", "pct_pos_total"]].head(8).to_string())

In [ ]:
# Какие теги похожи на «взрослые»? Просмотри список и поправь ADULT_TAGS ниже
import re, collections

def to_list(v):
    """Разбирает значение в список: поддерживает "['A', 'B']", "A, B" и "A;B"."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    if s[0] in "[{":
        try:
            import ast
            parsed = ast.literal_eval(s)
            if isinstance(parsed, dict):
                return [str(k).strip() for k in parsed.keys()]
            return [str(x).strip() for x in parsed]
        except Exception:
            s = s.strip("[]{}")
    return [p.strip(" '\"") for p in re.split(r"[;,]", s) if p.strip(" '\"")]

if "tags" in raw:
    cnt = collections.Counter(t for v in raw["tags"].dropna().head(30000) for t in to_list(v))
    print([ (t, n) for t, n in cnt.items() if re.search(r"sex|nud|hentai|adult|erotic|nsfw|gore|mature|porn", t, re.I)])
print("Топ-30 жанров:", collections.Counter(g for v in raw["genres"].dropna().head(30000) for g in to_list(v)).most_common(30))

> Имена столбцов уже сверены с файлом `games_march2025_full.csv` (47 столбцов). Теперь можно запускать всё подряд.
> **Пришли:** вывод раздела 2, воронку отбора (раздел 4–5) и «Итоговую сводку» в самом конце.

## 3. Настройки: правила отбора

Все «ручки» собраны в одном месте, чтобы воронка отбора была воспроизводимой и её можно было описать в записке.

In [ ]:
SEED = 42                 # воспроизводимость выборки
TARGET_N = 1500           # итоговый размер выборки (требование: 1–2 тыс. игр)
MIN_REVIEWS = 50          # меньше отзывов — доля положительных ничего не значит
BALANCE_WARN = 10         # расхождение индекса баланса отзывов (п.), выше которого игру считаем «спорной» для контроля
MAX_AGE = 17              # required_age >= 18 исключаем
MIN_YEAR, MAX_YEAR = 2009, 2024   # решение руководителя (Д5); набор собран в марте 2025: 2025 год неполный

# Теги для взрослых (проверь по списку из раздела 2)
ADULT_TAGS = {"Sexual Content", "Nudity", "Hentai", "NSFW", "Erotic", "Mature", "Adult Only"}
# В столбце notes разработчики пишут предупреждения о содержимом — ловим «взрослые» слова
NOTE_ADULT_RE = r"\b(?:sexual|nudity|nude|erotic|hentai|adult only|18\+)\b"

# Ценовые диапазоны, USD; 0 — бесплатные (отдельный диапазон)
PRICE_BINS = [0, 0.01, 5, 10, 20, 40, float("inf")]
PRICE_LABELS = ["Бесплатно", "до 5", "5–10", "10–20", "20–40", "от 40"]

# Если в наборе нет ссылок на обложки, строим их по appid (так устроены картинки магазина Steam).
# Ссылку обязательно проверь в браузере на 3–4 играх; недоступные обложки закроет заглушка на сайте.
COVER_TEMPLATE = "https://cdn.akamai.steamstatic.com/steam/apps/{appid}/header.jpg"

# В наборе есть не только игры, но и программы (Steam продаёт и их): на сайте о видеоиграх им не место
SOFTWARE_GENRES = {"Utilities", "Design & Illustration", "Animation & Modeling", "Game Development",
                   "Web Publishing", "Audio Production", "Software Training", "Video Production",
                   "Photo Editing", "Accounting", "Education", "Movie"}
# В списке «жанров» лежат и статусы/дескрипторы — это не жанры, в фильтр и графики по жанрам не берём
NOT_GENRES = {"Free To Play", "Early Access", "Violent", "Gore", "Sexual Content", "Nudity"}

## 4. Очистка и воронка отбора

In [ ]:
import numpy as np

funnel = []
def step(label, df):
    funnel.append((label, len(df)))
    print(f"{len(df):>8}  {label}")
    return df

df = step("Исходные записи", raw.copy())

df = step("Без дубликатов по appid", df.drop_duplicates("appid"))
df["name"] = df["name"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
df = step("Есть название", df[df["name"].notna() & (df["name"] != "")])
# Одинаковое название у нескольких appid (издания, дубли): оставляем запись с наибольшим числом отзывов
if "num_reviews_total" in df.columns:
    df = df.sort_values("num_reviews_total", ascending=False, kind="stable", na_position="last")
df = step("Без дубликатов по названию (оставлена запись с большим числом отзывов)", df.drop_duplicates("name"))

# Дата → ISO ГГГГ-ММ-ДД и год. «Скоро выйдет» и пустые уйдут в NaT
df["date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = step("Есть распознаваемая дата выхода", df[df["date"].notna()])
df["year"] = df["date"].dt.year
df = step(f"Год выхода {MIN_YEAR}–{MAX_YEAR}", df[df["year"].between(MIN_YEAR, MAX_YEAR)])

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = step("Есть цена", df[df["price"].notna() & (df["price"] >= 0)])

# Решение руководителя (Д3): все игры с ценой 0 считаем бесплатными, отдельного отсева нет
print(f"          (бесплатных, т.е. с ценой 0: {int((df['price'] == 0).sum())})")

# Обложка: берём ссылку из набора (только https), иначе строим по appid
built = pd.Series([COVER_TEMPLATE.format(appid=a) for a in df["appid"]], index=df.index)
if "header_image" in df.columns:
    own = df["header_image"].astype("string")
    df["cover"] = own.where(own.str.startswith("https", na=False), built)
else:
    df["cover"] = built
df = step("Обложка: ссылка из набора или по шаблону от appid", df)

# Отзывы (решение руководителя Д1): число отзывов и порог — по num_reviews_total,
# доля положительных ПЕРЕСЧИТЫВАЕТСЯ по positive и negative. Для контроля расхождения
# используем индекс баланса отзывов: (positive - negative) / (positive + negative) * 100, от -100 до +100.
if not ({"positive", "negative"} <= set(df.columns) and "num_reviews_total" in df.columns):
    raise KeyError("Нужны столбцы positive, negative и num_reviews_total — пришли список столбцов.")
tot = df["positive"] + df["negative"]
df["pct_pos"] = (df["positive"] / tot * 100).where(tot > 0)
df["balance"] = ((df["positive"] - df["negative"]) / tot * 100).where(tot > 0)
if "pct_pos_total" in df.columns:
    bal_steam = 2 * df["pct_pos_total"] - 100           # тот же индекс, но из готовой доли pct_pos_total
    gap = (df["balance"] - bal_steam).abs()
    print(f"          (индекс баланса: медиана расхождения pct_pos_total и пересчёта {gap.median():.2f} п., "
          f"расхождение > {BALANCE_WARN} п. у {int((gap > BALANCE_WARN).sum())} игр)")
    no_calc = df["pct_pos"].isna() & df["pct_pos_total"].notna()
    print(f"          (нет positive/negative, но есть pct_pos_total: {int(no_calc.sum())} игр — они уйдут из выборки, "
          "чтобы все значения считались одним способом)")
df["pct_pos"] = df["pct_pos"].round(1)
df["balance"] = df["balance"].round(1)
df["n_reviews"] = df["num_reviews_total"]
df = step("Есть positive/negative для пересчёта доли", df[df["pct_pos"].notna()])
df = step(f"Не менее {MIN_REVIEWS} отзывов", df[df["n_reviews"] >= MIN_REVIEWS])

# 18+ и «взрослые» теги
age_ok = df["required_age"].fillna(0) <= MAX_AGE
df = step("Без возрастного ограничения 18+", df[age_ok])
if "tags" in df:
    tagsets = df["tags"].map(lambda v: set(to_list(v)))
    for t in sorted(ADULT_TAGS):
        print(f"          тег {t}: {int(tagsets.map(lambda x, t=t: t in x).sum())} игр")
    only_mature = tagsets.map(lambda x: "Mature" in x and not ((ADULT_TAGS - {"Mature"}) & x))
    print(f"          из них уйдут ТОЛЬКО из-за Mature: {int(only_mature.sum())} "
          "(если их много и среди них обычные игры 16+, убери Mature из ADULT_TAGS)")
    is_adult = tagsets.map(lambda x: bool(ADULT_TAGS & x))
    df = step("Без тегов для взрослых", df[~is_adult])
else:
    print("          (столбца tags нет: фильтр по тегам НЕ применён — в наборе возраст указан не у всех игр, "
          "поэтому найди другой столбец для защиты от 18+ и скажи мне)")
if "notes" in df:
    note_adult = df["notes"].astype("string").str.contains(NOTE_ADULT_RE, case=False, na=False, regex=True)
    df = step("Без «взрослых» пометок в notes", df[~note_adult])
if "genres" in df:
    is_adult_g = df["genres"].map(lambda v: bool({"Sexual Content", "Nudity"} & set(to_list(v))))
    df = step("Без «взрослых» жанров", df[~is_adult_g])
    is_soft = df["genres"].map(lambda v: bool(SOFTWARE_GENRES & set(to_list(v))))
    df = step("Без программ, не являющихся играми", df[~is_soft])

df["price_bin"] = pd.cut(df["price"], bins=PRICE_BINS, labels=PRICE_LABELS, right=False)
df["price_bin"] = df["price_bin"].astype("string")
universe = df.copy()   # все игры, прошедшие фильтры (до стратификации): по ним считаем графики по годам и жанрам (Д6)

## 5. Стратифицированная выборка

Чтобы в выборку попали не только самые популярные игры, берём не больше `cap` игр
из каждой группы «год выхода × ценовой диапазон». Порог `cap` подбирается бинарным поиском так,
чтобы получилось около `TARGET_N` игр.

In [ ]:
groups = df.groupby(["year", "price_bin"], observed=True)
sizes = groups.size()

def total_for(cap):
    return int(np.minimum(sizes.values, cap).sum())

lo, hi = 1, int(sizes.max())
while lo < hi:
    mid = (lo + hi) // 2
    if total_for(mid) < TARGET_N:
        lo = mid + 1
    else:
        hi = mid
cap = lo
print("cap на группу:", cap, "→ ожидаемо игр:", total_for(cap), "в", len(sizes), "группах")

# Перемешиваем (с фиксированным seed), нумеруем игры внутри группы и берём первые cap.
# Так работает во всех версиях pandas (groupby.apply в новых версиях теряет столбцы группировки).
shuffled = df.sample(frac=1, random_state=SEED)
shuffled["_rank"] = shuffled.groupby(["year", "price_bin"], observed=True).cumcount()
sample = shuffled[shuffled["_rank"] < cap].drop(columns="_rank")
sample = sample.sort_values("appid").reset_index(drop=True)
funnel.append(("Стратифицированная выборка", len(sample)))
print("Итог выборки:", len(sample))

## 6. Признаки и агрегаты

In [ ]:
def has_ru(v):
    return any(l.lower().startswith("russian") for l in to_list(v))

sample["ru"] = sample["supported_languages"].map(has_ru)
sample["ru_audio"] = sample["full_audio_languages"].map(has_ru) if "full_audio_languages" in sample else False
sample["genres_list"] = sample["genres"].map(lambda v: [g for g in to_list(v) if g not in NOT_GENRES])

# Краткое описание для карточки игры: без HTML, не длиннее ~220 символов (иначе вырастет JSON)
import html

def short_text(v, limit=220):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = re.sub(r"<[^>]+>", " ", html.unescape(str(v)))
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= limit:
        return s or None
    return s[:limit].rsplit(" ", 1)[0] + "…"

sample["desc"] = sample["short_description"].map(short_text) if "short_description" in sample else None

# До трёх скриншотов (только https); ссылки, а не файлы
def https_urls(v, limit=3):
    return [u for u in to_list(v) if u.startswith("https")][:limit]

sample["shots_list"] = (sample["screenshots"].map(https_urls) if "screenshots" in sample
                        else [[] for _ in range(len(sample))])
if "discount" not in sample:
    sample["discount"] = np.nan

def truthy(v):
    return str(v).strip().lower() in ("true", "1")

def platforms_of(r):
    p = []
    if truthy(r.get("windows")): p.append("Windows")
    if truthy(r.get("mac")): p.append("macOS")
    if truthy(r.get("linux")): p.append("Linux")
    return p
sample["platforms_list"] = sample.apply(platforms_of, axis=1)

# Для калькулятора: среднее время игры (минуты → часы), если поле есть
if "average_playtime_forever" in sample:
    sample["playtime_h"] = (sample["average_playtime_forever"] / 60).round(1).where(sample["average_playtime_forever"] > 0)
else:
    sample["playtime_h"] = np.nan

# Размеры ценовых диапазонов и медианы — по ВЫБОРКЕ
prices_agg = (sample.groupby("price_bin", observed=True)
                .agg(n=("appid", "size"), median_pct_pos=("pct_pos", "median"))
                .reindex(PRICE_LABELS).dropna().round(1))
print(prices_agg)

# Д6: агрегаты по ВСЕМ играм, прошедшим фильтры (не по выборке) — для графиков «по годам» и «по жанрам»
by_year = (universe.groupby("year")
             .agg(n=("appid", "size"), median_pct_pos=("pct_pos", "median"), median_price=("price", "median"))
             .round(1).reset_index())
gen = universe[["appid", "genres", "pct_pos", "price"]].copy()
gen["genre"] = gen["genres"].map(lambda v: [g for g in to_list(v) if g not in NOT_GENRES])
gen = gen.explode("genre").dropna(subset=["genre"])
by_genre = (gen.groupby("genre")
              .agg(n=("appid", "nunique"), median_pct_pos=("pct_pos", "median"), median_price=("price", "median"))
              .round(1).sort_values("n", ascending=False).head(15).reset_index())
print("Всего игр для агрегатов:", len(universe))
print(by_year.to_string(index=False))
print(by_genre.to_string(index=False))
print("Диапазоны с n < 30 объедини или оговори на сайте:", prices_agg[prices_agg["n"] < 30].index.tolist())

## 7. Экспорт JSON и контроль

In [ ]:
import json, math

def clean(v):
    """В JSON не должно быть NaN: пропуски → null."""
    if v is None: return None
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)):
        return None if math.isnan(v) else round(float(v), 2)
    if isinstance(v, (np.bool_,)): return bool(v)
    if v is pd.NA: return None
    return v

games = []
for r in sample.itertuples(index=False):
    games.append({
        "id": clean(r.appid),
        "name": clean(r.name),
        "date": r.date.strftime("%Y-%m-%d"),
        "year": int(r.year),
        "price": clean(r.price),
        "priceBin": clean(r.price_bin),
        "platforms": r.platforms_list,
        "ru": bool(r.ru),
        "ruAudio": bool(r.ru_audio),
        "genres": r.genres_list,
        "pctPos": clean(r.pct_pos),
        "balance": clean(r.balance),
        "reviews": clean(r.n_reviews),
        "playtimeH": clean(r.playtime_h),
        "cover": clean(r.cover),
        "desc": r.desc if isinstance(r.desc, str) else None,
        "shots": r.shots_list,
        "discount": clean(r.discount),
    })

prices = {
    # по выборке (стратифицирована по году и цене: числа игр в диапазонах — следствие метода)
    "bins": [{"bin": b, "n": int(row.n), "medianPctPos": clean(row.median_pct_pos)}
             for b, row in prices_agg.iterrows()],
    # по ВСЕМ играм, прошедшим фильтры (не по выборке), набор собран в марте 2025
    "universeSize": int(len(universe)),
    "byYear": [{"year": int(r.year), "n": int(r.n), "medianPctPos": clean(r.median_pct_pos),
                "medianPrice": clean(r.median_price)} for r in by_year.itertuples()],
    "byGenre": [{"genre": r.genre, "n": int(r.n), "medianPctPos": clean(r.median_pct_pos),
                 "medianPrice": clean(r.median_price)} for r in by_genre.itertuples()],
}

plat_counts = {p: int(sample["platforms_list"].map(lambda l, p=p: p in l).sum()) for p in ["Windows", "macOS", "Linux"]}
ru_by_year = (sample.groupby("year")["ru"].agg(["size", "mean"]).round(3)
                .reset_index().rename(columns={"size": "n", "mean": "shareRu"}))
med_ru = sample.groupby("ru")["pct_pos"].median().round(1).to_dict()
platforms = {
    "counts": plat_counts,
    "ruByYear": [{"year": int(r.year), "n": int(r.n), "shareRu": clean(r.shareRu)} for r in ru_by_year.itertuples()],
    "medianPctPos": {"withRu": clean(med_ru.get(True)), "withoutRu": clean(med_ru.get(False))},
}

os.makedirs("data", exist_ok=True)
for fname, obj in [("games.json", games), ("prices.json", prices), ("platforms.json", platforms)]:
    with open(f"data/{fname}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, separators=(",", ":"), allow_nan=False)  # NaN/Infinity -> сразу ошибка
    size = os.path.getsize(f"data/{fname}") / 1024 / 1024
    print(f"data/{fname}: {size:.2f} МБ", "(ОК)" if size <= 2 else "(ВЕЛИК — убери поля или игры)")

In [ ]:
# Контроль качества
assert 1000 <= len(games) <= 2000, f"Размер выборки вне 1–2 тыс.: {len(games)}"
def _forbid(token):
    raise ValueError(f"в JSON найден недопустимый токен {token}")

for fname in ["games.json", "prices.json", "platforms.json"]:
    # parse_constant срабатывает только на настоящие NaN/Infinity, а не на слова внутри названий и описаний
    with open(f"data/{fname}", encoding="utf-8") as f:
        json.load(f, parse_constant=_forbid)
assert len({g["id"] for g in games}) == len(games), "Дубли id"
assert all(g["cover"].startswith("https") for g in games), "Есть обложки не по https"
assert all(g["reviews"] >= MIN_REVIEWS for g in games), "Есть игры меньше порога отзывов"
print("Все проверки пройдены")

print("\nВОРОНКА ОТБОРА (для параграфа 1.3 и Приложения А):")
for label, n in funnel:
    print(f"{n:>8}  {label}")

In [ ]:
# Итоговая сводка — скопируй вывод этой ячейки и пришли
print("Файл набора:", os.path.basename(SRC))
print("Игр в выборке:", len(games))
print("Со скидкой (discount > 0):", int((sample["discount"].fillna(0) > 0).sum()))
print("С описанием:", sum(1 for g in games if g["desc"]))
print("Со скриншотами:", sum(1 for g in games if g["shots"]))
print("Со временем игры (playtimeH):", sum(1 for g in games if g["playtimeH"] is not None))
print("С русским языком:", f"{sample['ru'].mean():.0%}")
print("Бесплатных:", int((sample["price"] == 0).sum()))
print("По годам:", sample["year"].value_counts().sort_index().to_dict())
print("Платформы:", plat_counts)
print("Пример записи:")
print(json.dumps(games[0], ensure_ascii=False)[:900])

In [ ]:
# Скачать результат (для загрузки в папку data/ репозитория)
from google.colab import files
for fname in ["games.json", "prices.json", "platforms.json"]:
    files.download(f"data/{fname}")